# Phase 9: Cross Validation

Time-series cross validation using the Phase 8 feature pipeline and selected model.

## 1. Setup

In [1]:
import sys
import pickle
from pathlib import Path

import pandas as pd
from sklearn.base import clone
from sklearn.model_selection import TimeSeriesSplit

# Resolve project root whether executed from repo root or notebooks/ via nbconvert
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent.resolve()

src_path = str(PROJECT_ROOT / 'src')
if src_path not in sys.path:
    sys.path.insert(0, src_path)

from models import prepare_timeseries_data, get_feature_names
from evaluate import evaluate_model

DATA_PROCESSED = PROJECT_ROOT / 'data' / 'processed'
DATA_OUTPUTS = PROJECT_ROOT / 'data' / 'outputs'
REPORTS_TABLES = PROJECT_ROOT / 'reports' / 'tables'
MODELS_DIR = PROJECT_ROOT / 'models'

REPORTS_TABLES.mkdir(parents=True, exist_ok=True)

print('Setup complete')

Setup complete


## 2. Load Data and Model

In [2]:
df = pd.read_csv(DATA_PROCESSED / 'feature_ready_dataset.csv')

if 'Week_End_Date' not in df.columns:
    raise ValueError('Week_End_Date is required for Phase 9 time-series cross validation')

df['Week_End_Date'] = pd.to_datetime(df['Week_End_Date'])
df = df.sort_values('Week_End_Date').reset_index(drop=True)

best_models = pd.read_csv(DATA_OUTPUTS / 'best_models.csv')
model_file = best_models.loc[0, 'Model_File']
model_path = MODELS_DIR / model_file

with open(model_path, 'rb') as f:
    base_model = pickle.load(f)

feature_cols = get_feature_names(df)

print(f'Data shape: {df.shape}')
print(f'Date range: {df["Week_End_Date"].min().date()} to {df["Week_End_Date"].max().date()}')
print(f'Model template: {type(base_model).__name__}')
print(f'Feature count: {len(feature_cols)}')
print(f'Features: {feature_cols}')

Data shape: (2600, 30)
Date range: 2023-01-07 to 2024-12-28
Model template: GradientBoostingRegressor
Feature count: 23
Features: ['Year', 'Month', 'Quarter', 'ISO_Week', 'Week_Number', 'Lag_1', 'Lag_2', 'Lag_3', 'Lag_4', 'Lag_8', 'Lag_12', 'Rolling_Mean_4', 'Rolling_Mean_8', 'Rolling_Std_4', 'Rolling_Max_4', 'Rolling_Min_4', 'Discount_Flag', 'Returns_Rate', 'Inventory_Ratio', 'Promo_Flag', 'Holiday_Flag', 'Rainfall', 'Avg_Temperature']


## 3. TimeSeriesSplit Cross Validation

In [3]:
tscv = TimeSeriesSplit(n_splits=3)
fold_rows = []

for fold, (train_idx, val_idx) in enumerate(tscv.split(df), start=1):
    train_df = df.iloc[train_idx]
    val_df = df.iloc[val_idx]
    
    X_train, y_train = prepare_timeseries_data(train_df)
    X_val, y_val = prepare_timeseries_data(val_df)
    
    model = clone(base_model)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_val)
    
    metrics = evaluate_model(y_val, y_pred)
    fold_rows.append({
        'Fold': fold,
        'Train_Start': train_df['Week_End_Date'].min().date(),
        'Train_End': train_df['Week_End_Date'].max().date(),
        'Validation_Start': val_df['Week_End_Date'].min().date(),
        'Validation_End': val_df['Week_End_Date'].max().date(),
        'Train_Rows': len(train_df),
        'Validation_Rows': len(val_df),
        'MAE': metrics['MAE'],
        'RMSE': metrics['RMSE'],
        'WAPE': metrics['WAPE']
    })

fold_metrics = pd.DataFrame(fold_rows)
fold_metrics

,Fold,Train_Start,Train_End,Validation_Start,Validation_End,Train_Rows,Validation_Rows,MAE,RMSE,WAPE
0,1,2023-01-07,2023-07-01,2023-07-08,2023-12-30,650,650,5.886356,6.438666,10.649738
1,2,2023-01-07,2023-12-30,2024-01-06,2024-06-29,1300,650,11.246000,11.867589,19.611257
2,3,2023-01-07,2024-06-29,2024-07-06,2024-12-28,1950,650,9.216233,9.973016,16.493355


## 4. Average Metrics

In [4]:
metric_cols = ['MAE', 'RMSE', 'WAPE']

cross_validation_summary = pd.DataFrame({
    'Metric': metric_cols,
    'Average': [fold_metrics[col].mean() for col in metric_cols],
    'Std': [fold_metrics[col].std() for col in metric_cols],
    'Min': [fold_metrics[col].min() for col in metric_cols],
    'Max': [fold_metrics[col].max() for col in metric_cols]
})

cross_validation_summary

,Metric,Average,Std,Min,Max
0,MAE,8.782863,2.705975,5.886356,11.246000
1,RMSE,9.426424,2.755426,6.438666,11.867589
2,WAPE,15.584784,4.549322,10.649738,19.611257


## 5. Save Outputs

In [5]:
fold_metrics_path = REPORTS_TABLES / 'fold_metrics.csv'
summary_path = REPORTS_TABLES / 'cross_validation_summary.csv'

fold_metrics.to_csv(fold_metrics_path, index=False)
cross_validation_summary.to_csv(summary_path, index=False)

print(f'Saved: {fold_metrics_path}')
print(f'Saved: {summary_path}')

Saved: D:\Final Project\reports\tables\fold_metrics.csv
Saved: D:\Final Project\reports\tables\cross_validation_summary.csv


## Phase 9 Complete